In [79]:
import pandas as pd
import os
import numpy as np
import astropy
import astropy.units as u
from astropy.constants import M_jup, M_earth
import astropy.units as u
import matplotlib.pyplot as plt

def gen_orbits_csv(k,  output_dir, output_file, filter_snr = False,):
    '''
    Generates a CSV file with a synthetic population of exoplanet orbits.

    Draws orbital elements and physical properties for k planets.
    Two modes are available:
      - Agnostic (filter_snr=False): uniform draws for all parameters
        (sma in [1, 100] AU, mass in [M_mars, 80] Mjup, ecc in [0, 0.99],
        isotropic inclination, uniform angles).
      - SNR-filtered (filter_snr=True): importance samples sma and mass
        using a Gaia DR4 astrometric SNR proxy with
        a 24.75-year baseline, then applies rejection sampling. Yields a
        population biased toward detectable orbits.

    Stellar masses are drawn uniformly in [0.5, 1.5] Msun for both modes. (FGK)

    Parameters
    ----------
    k : int
        Number of orbits to propose. Equal to the final population size
        when filter_snr=False; acts as the proposal pool size when
        filter_snr=True (accepted count will be a lot smaller D: ).
    output_dir : str
        Directory for the output CSV. Created if it doesn't exist.
    output_file : str
        Filename for the output CSV.
    filter_snr : bool, optional
        If True, apply SNR-based rejection sampling. Default is False.

    Returns
    -------
    csv_data : pandas.DataFrame
        DataFrame with columns: sma, ecc, mass_pl, mass_st, inc,
        Omega, omega, M_anom.
    '''
    k_proposal = int(k)
    M_mars_Mjup = (0.107 * M_earth).to(u.M_jup).value
    rng = np.random.default_rng(42)
    os.makedirs(output_dir, exist_ok=True)

    # Filters sma to snr sensitive values for gaia dr4
    if filter_snr == True:
        T_BASELINE = 24.75
        # Draw stellar masses (FGK), uniform
        mstar = rng.uniform(0.5, 1.5, size=k_proposal)
        # SMA_CRIT for a given star
        sma_crit = (T_BASELINE**2 * mstar)**(1/3)
        # Log-uniform draws in sma and mass
        sma_prop = 10**rng.uniform(np.log10(1), np.log10(100), size=k_proposal)
        mass_prop = 10**rng.uniform(np.log10(M_mars_Mjup), np.log10(80), size=k_proposal)
        # SNR proxy from William's eqn
        snr_proxy = (mass_prop / mstar) * sma_prop / (1.0 + (sma_prop / sma_crit)**3)
        accept_prob = snr_proxy / snr_proxy.max()
        accept = rng.uniform(size=k_proposal) < accept_prob
        sma = sma_prop[accept]
        mass = mass_prop[accept]
        mstar = mstar[accept]
        n = accept.sum()
        print(f"Accepted {n} / {k_proposal} = {n/k_proposal:.1%}")
        ecc = rng.uniform(0, 0.99, size=n)
        inc = np.degrees(np.arccos(rng.uniform(-1, 1, size=n)))
        Omega = rng.uniform(0, 360, size=n)
        omega = rng.uniform(0, 360, size=n)
        M_anom = rng.uniform(0, 360, size=n)

    # Completely agnostic - everything is uniform
    else:
        ecc = rng.uniform(0, 0.99, size=k_proposal)
        inc = np.degrees(np.arccos(rng.uniform(-1, 1, size=k_proposal)))
        Omega = rng.uniform(0, 360, size=k_proposal)
        omega = rng.uniform(0, 360, size=k_proposal)
        M_anom = rng.uniform(0, 360, size=k_proposal)
        sma = rng.uniform((1), (100), size=k_proposal)
        mass = rng.uniform((M_mars_Mjup), (80), size=k_proposal)
        mstar = rng.uniform(0.5, 1.5, size=k_proposal)

    labeled_data = {
        'sma': sma,
        'ecc': ecc,
        'mass_pl': mass,
        'mass_st': mstar,
        'inc': inc,
        'Omega': Omega,
        'omega': omega,
        'M_anom': M_anom,
    }
    csv_data = pd.DataFrame(labeled_data)
    output_fpath = os.path.join(output_dir, output_file)
    print(f"Writing output to {output_fpath}...")
    csv_data.to_csv(output_fpath, index=False)
    return csv_data
output_dir_name = 'outputs/'
gen_orbits_csv(int(1e5), output_dir = output_dir_name, output_file = 'agnostic_pop.csv', filter_snr = False)

Writing output to outputs/agnostic_pop.csv...


,sma,ecc,mass_pl,mass_st,inc,Omega,omega,M_anom
0,36.695025,0.766216,43.320749,0.898469,28.628295,145.329862,172.558796,143.885350
1,5.986216,0.434490,28.675390,1.296078,26.460142,141.044015,148.110135,320.538777
2,97.435030,0.850012,57.209458,0.855785,85.870343,312.483297,275.991923,175.388615
3,57.587875,0.690394,56.116234,0.682901,119.885625,207.941467,266.640141,261.509087
4,77.900423,0.093236,53.099415,1.410529,61.402978,273.580544,269.572201,143.220099
...,...,...,...,...,...,...,...,...
99995,74.777871,0.263885,46.851494,1.226762,129.675121,108.350281,194.294369,271.431480
99996,98.734392,0.569274,59.597131,0.644788,75.967749,64.053064,6.594289,65.050878
99997,60.897321,0.554404,38.342024,0.698014,127.064459,178.833128,60.215781,285.765114
99998,19.223302,0.119816,55.356445,1.188566,141.347731,38.254747,46.900359,332.315113
